# CICIDS2017_TensorFlow_MLP\n\nSplit Kaggle notebook generated from `02_CICIDS2017_TensorFlow.ipynb`. This notebook runs only the MLP baseline model.\n

## 2. Install/import dependencies

Colab already includes the core scientific Python stack. The install cell is intentionally minimal and can be extended if your runtime is missing a package.

In [ ]:
# Colab usually has these installed. Uncomment if your runtime is missing packages.
# !pip install -q pandas numpy scikit-learn tensorflow

In [ ]:
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISM"] = "1"

import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATASET_NAME = "CICIDS-2017"
FRAMEWORK = "TensorFlow/Keras"
NOTEBOOK_FILENAME = "CICIDS2017_TensorFlow_MLP.ipynb"
FRAMEWORK_STYLE_MODEL_NAME = "INSOMNIA-style MLP"
IMPROVED_MODEL_NAME = "Improved CICIDS-2017 MLP"


## 3. Set reproducibility configuration

The training seeds and weight initialization tuples follow the reproducibility setup from “Randomness Unmasked: Towards Reproducible and Fair Evaluation of Shift-Aware Deep Learning NIDS”.

In [ ]:
TRAINING_SEEDS = [57, 305, 5, 9667, 405]
WEIGHT_INIT_SEEDS = {
    "W1": [1004, 77, 259, 35],
    "W2": [8, 358, 200, 35],
    "W3": [487, 22, 900, 7],
}

DEFAULT_SEED = TRAINING_SEEDS[0]

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_global_seed(DEFAULT_SEED)

## 4. Create data and Kaggle results folders\n\nGenerated CSV results are written to `/kaggle/working/results/`.\n

In [ ]:
DATA_DIR = Path("/content/data")
RESULTS_DIR = Path("/kaggle/working/results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
RESULTS_CSV_PATH = RESULTS_DIR / "CICIDS2017_TensorFlow_MLP_results.csv"
AGGREGATED_RESULTS_CSV_PATH = RESULTS_DIR / "CICIDS2017_TensorFlow_MLP_aggregated_results.csv"

print("Per-run results CSV:", RESULTS_CSV_PATH)
print("Aggregated results CSV:", AGGREGATED_RESULTS_CSV_PATH)


## 5. Dataset download with kagglehub

This notebook downloads the Kaggle dataset with `kagglehub`, copies the downloaded files into `/content/data`, preserves subdirectories, and prints discovered CSV files.

In [ ]:
# Colab shell equivalent: !pip install -q kagglehub
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

In [ ]:
import shutil
import kagglehub

path = kagglehub.dataset_download("bertvankeulen/cicids-2017")
print("Path to dataset files:", path)

source_path = Path(path)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if source_path.is_file():
    target_path = DATA_DIR / source_path.name
    if source_path.resolve() != target_path.resolve():
        shutil.copy2(source_path, target_path)
else:
    for source_item in source_path.rglob("*"):
        if source_item.is_file():
            relative_path = source_item.relative_to(source_path)
            target_path = DATA_DIR / relative_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            if source_item.resolve() != target_path.resolve():
                shutil.copy2(source_item, target_path)

csv_files = sorted(DATA_DIR.rglob("*.csv"))
print(f"CSV files discovered after copying: {len(csv_files)}")
for csv_file in csv_files:
    print(csv_file)

## 6. Load dataset from /content/data

This loader recursively finds CSV files with `Path("/content/data").rglob("*.csv")`, adds a `source_file` column before concatenation, prints the discovered files, reports dataset shape, and prints label distribution.

In [ ]:
LABEL_CANDIDATES = ["Label", "label", "Class", "class", "Attack", "attack"]
CICIDS_ORIGINAL_WEEKDAYS = ("monday", "tuesday", "wednesday", "thursday", "friday")


def detect_label_column(df: pd.DataFrame) -> str:
    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate

    lower_map = {col.lower(): col for col in df.columns}
    for candidate in LABEL_CANDIDATES:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    object_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    object_columns = [col for col in object_columns if col != "source_file"]
    if object_columns:
        return object_columns[-1]

    raise ValueError(
        "Could not automatically detect a label column. Rename the target column to one of: "
        + ", ".join(LABEL_CANDIDATES)
    )

def load_csv_dataset(data_dir: Path) -> pd.DataFrame:
    csv_files = sorted(data_dir.rglob("*.csv"))
    csv_files = [
        csv_path
        for csv_path in csv_files
        if "_plus" not in csv_path.name.lower()
        and any(day in csv_path.name.lower() for day in CICIDS_ORIGINAL_WEEKDAYS)
    ]
    print(f"CSV files found after excluding *_plus.csv files: {len(csv_files)}")
    print("CSV files used:")
    for csv_path in csv_files:
        print(" -", csv_path.relative_to(data_dir))

    if not csv_files:
        raise FileNotFoundError(
            "No CSV files found under /content/data. Check that kagglehub downloaded successfully "
            "and that files were copied into /content/data."
        )

    frames = []
    for csv_path in csv_files:
        print(f"Loading {csv_path}")
        frame = pd.read_csv(csv_path, low_memory=False)
        frame.columns = frame.columns.astype(str).str.strip()
        frame["source_file"] = str(csv_path.relative_to(data_dir))
        frames.append(frame)

    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    df.columns = df.columns.astype(str).str.strip()

    print(f"Dataset shape: {df.shape}")
    print("Column names:")
    print(list(df.columns))

    label_col = detect_label_column(df)
    print(f"Detected label column: {label_col}")
    print("Label distribution:")
    print(df[label_col].astype(str).str.strip().value_counts(dropna=False))
    return df

raw_df = load_csv_dataset(DATA_DIR)
raw_df.head()

## 7. Data preprocessing

Preprocessing is fit after the paper-style train/test protocol is built. The label encoder is fit on training labels when possible, and `StandardScaler` is fit only on `X_train` before transforming `X_test`.

In [ ]:
def clean_label_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()

def is_benign_label(label: str) -> bool:
    normalized = str(label).strip().lower()
    return normalized in {"benign", "normal", "normal traffic", "background"} or "benign" in normalized

def to_binary_attack_labels(labels: pd.Series) -> pd.Series:
    return clean_label_series(labels).apply(lambda value: "Benign" if is_benign_label(value) else "Attack")

def prepare_feature_matrix(train_df: pd.DataFrame, test_df: pd.DataFrame, label_col: str):
    X_train_df = train_df.drop(columns=[label_col], errors="ignore").copy()
    X_test_df = test_df.drop(columns=[label_col], errors="ignore").copy()

    for frame in (X_train_df, X_test_df):
        frame.drop(columns=["source_file"], inplace=True, errors="ignore")
        for col in frame.columns:
            if frame[col].dtype == "object":
                frame[col] = pd.to_numeric(frame[col].astype(str).str.strip(), errors="coerce")

    numeric_cols = X_train_df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        raise ValueError("No numeric feature columns were found after protocol splitting.")

    X_train_df = X_train_df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    X_test_df = X_test_df.reindex(columns=numeric_cols).replace([np.inf, -np.inf], np.nan)

    train_medians = X_train_df.median(numeric_only=True).fillna(0)
    X_train_df = X_train_df.fillna(train_medians).fillna(0)
    X_test_df = X_test_df.fillna(train_medians).fillna(0)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_df).astype(np.float32)
    X_test = scaler.transform(X_test_df).astype(np.float32)
    return X_train, X_test, scaler, numeric_cols

def fit_label_encoder_train_when_possible(y_train_raw: pd.Series, y_test_raw: pd.Series):
    y_train_raw = clean_label_series(y_train_raw)
    y_test_raw = clean_label_series(y_test_raw)
    train_classes = set(y_train_raw.unique())
    test_classes = set(y_test_raw.unique())
    unseen_test_classes = sorted(test_classes - train_classes)

    label_encoder = LabelEncoder()
    if unseen_test_classes:
        print("Test labels not present in training labels; fitting LabelEncoder on train+test labels for transform compatibility:")
        print(unseen_test_classes)
        label_encoder.fit(pd.concat([y_train_raw, y_test_raw], ignore_index=True))
    else:
        label_encoder.fit(y_train_raw)

    return label_encoder, label_encoder.transform(y_train_raw), label_encoder.transform(y_test_raw)

def prepare_protocol_data(train_df: pd.DataFrame, test_df: pd.DataFrame, *, binary_labels: bool = False, metadata: dict | None = None):
    global X_train, X_test, y_train, y_test, label_encoder, scaler, feature_columns
    global NUM_FEATURES, NUM_CLASSES, IS_BINARY, RUN_METADATA

    label_col = detect_label_column(pd.concat([train_df.head(1), test_df.head(1)], ignore_index=True))
    train_df = train_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])
    test_df = test_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])

    if binary_labels:
        y_train_raw = to_binary_attack_labels(train_df[label_col])
        y_test_raw = to_binary_attack_labels(test_df[label_col])
    else:
        y_train_raw = clean_label_series(train_df[label_col])
        y_test_raw = clean_label_series(test_df[label_col])

    X_train, X_test, scaler, feature_columns = prepare_feature_matrix(train_df, test_df, label_col)
    label_encoder, y_train, y_test = fit_label_encoder_train_when_possible(y_train_raw, y_test_raw)
    y_train = y_train.astype(np.int64)
    y_test = y_test.astype(np.int64)

    NUM_FEATURES = X_train.shape[1]
    NUM_CLASSES = len(label_encoder.classes_)
    IS_BINARY = NUM_CLASSES == 2
    RUN_METADATA = metadata.copy() if metadata else {}

    print("Train shape:", X_train.shape, y_train.shape)
    print("Test shape:", X_test.shape, y_test.shape)
    print(f"Classes ({NUM_CLASSES}): {list(label_encoder.classes_)}")
    print("Scaler fitted on training data only.")
    return X_train, X_test, y_train, y_test

## 8. Train/test split

CICIDS-2017 uses a paper-style day-based protocol for every model: the first two chronological days/files are training data and the remaining days/files are test data.

In [ ]:
import re

WEEKDAY_ORDER = {
    "monday": 0,
    "tuesday": 1,
    "wednesday": 2,
    "thursday": 3,
    "friday": 4,
    "saturday": 5,
    "sunday": 6,
}

def _source_file_sort_key(source_file: str):
    name = Path(source_file).name.lower()
    for weekday, rank in WEEKDAY_ORDER.items():
        if weekday in name:
            return (0, rank, name)

    date_match = re.search(r"(20\d{2})[-_ ]?(\d{1,2})[-_ ]?(\d{1,2})", name)
    if date_match:
        year, month, day = map(int, date_match.groups())
        return (1, year, month, day, name)

    number_match = re.search(r"(\d+)", name)
    if number_match:
        return (2, int(number_match.group(1)), name)

    return None

def build_cicids2017_day_split(df: pd.DataFrame):
    if "source_file" not in df.columns:
        raise ValueError("CICIDS-2017 day split requires a source_file column added during CSV loading.")

    source_files = sorted(df["source_file"].dropna().astype(str).unique())
    sort_keys = {source_file: _source_file_sort_key(source_file) for source_file in source_files}
    failed = [source_file for source_file, key in sort_keys.items() if key is None]
    if failed:
        raise ValueError(
            "Could not detect chronological day/order from source_file names. "
            f"Available source_file names: {source_files}"
        )

    ordered_files = sorted(source_files, key=lambda source_file: sort_keys[source_file])

    def group_key(source_file: str):
        key = sort_keys[source_file]
        return key[:-1]

    ordered_day_keys = []
    for source_file in ordered_files:
        key = group_key(source_file)
        if key not in ordered_day_keys:
            ordered_day_keys.append(key)

    if len(ordered_day_keys) < 3:
        raise ValueError(f"Need at least 3 source files/days for CICIDS-2017 protocol. Found: {ordered_files}")

    train_day_keys = set(ordered_day_keys[:2])
    train_files = [source_file for source_file in ordered_files if group_key(source_file) in train_day_keys]
    test_files = [source_file for source_file in ordered_files if group_key(source_file) not in train_day_keys]
    print("CICIDS-2017 day-based split")
    print("Selected train files:")
    for file_name in train_files:
        print(" -", file_name)
    print("Selected test files:")
    for file_name in test_files:
        print(" -", file_name)

    train_df = df[df["source_file"].isin(train_files)].copy()
    test_df = df[df["source_file"].isin(test_files)].copy()
    return train_df, test_df, train_files, test_files

train_df, test_df, train_files, test_files = build_cicids2017_day_split(raw_df)
prepare_protocol_data(
    train_df,
    test_df,
    binary_labels=False,
    metadata={
        "held_out_attack": None,
        "train_files": json.dumps(train_files),
        "test_files": json.dumps(test_files),
    },
)

## 9. Define evaluation metrics

Metrics use scikit-learn. Multi-class precision, recall, and F1 use `average="weighted"`; binary experiments use binary averaging.

In [ ]:
def metric_average(num_classes: int) -> str:
    return "binary" if num_classes == 2 else "weighted"

def evaluate_predictions(y_true, y_pred, labels=None) -> dict:
    average = metric_average(NUM_CLASSES)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    try:
        metrics["classification_report"] = classification_report(
            y_true,
            y_pred,
            target_names=labels,
            zero_division=0,
            output_dict=True,
        )
    except Exception:
        metrics["classification_report"] = classification_report(
            y_true,
            y_pred,
            zero_division=0,
            output_dict=True,
        )
    return metrics

def _aggregate_with_group_cols(per_run_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    metric_cols = ["accuracy", "precision", "recall", "f1"]
    aggregated = per_run_df.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "min", "max", "std"]).round(6)
    aggregated.columns = [f"{metric}_{stat}" for metric, stat in aggregated.columns]
    return aggregated.reset_index()

def aggregate_results(per_run_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["dataset_name", "framework", "model_name"]
    if "held_out_attack" in per_run_df.columns and per_run_df["held_out_attack"].notna().any():
        per_attack = _aggregate_with_group_cols(per_run_df, base_cols + ["held_out_attack"])
        overall = _aggregate_with_group_cols(per_run_df, base_cols)
        overall.insert(len(base_cols), "held_out_attack", "OVERALL")
        return pd.concat([per_attack, overall], ignore_index=True)
    return _aggregate_with_group_cols(per_run_df, base_cols)

## 10. Define weight initialization utilities

Every model is trained with each of the three named weight initialization tuples. These utilities also define the common training loop used by all model sections.

In [ ]:
def initializer_from_seed(seed_value: int):
    return keras.initializers.GlorotUniform(seed=seed_value)

def weight_seed(seed_tuple, index: int) -> int:
    return int(seed_tuple[index % len(seed_tuple)])

def compile_model(model, lr: float = 1e-3):
    loss = keras.losses.BinaryCrossentropy(from_logits=True) if IS_BINARY else keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=loss)
    return model

def predict_tf(model, X_values: np.ndarray, threshold: float = 0.5):
    logits = model.predict(X_values, verbose=0)
    if IS_BINARY:
        probabilities = tf.sigmoid(logits).numpy().reshape(-1)
        return (probabilities >= threshold).astype(int), probabilities
    probabilities = tf.nn.softmax(logits, axis=1).numpy()
    return probabilities.argmax(axis=1), probabilities

def tune_binary_threshold(y_true, probabilities):
    if not IS_BINARY:
        return 0.5
    thresholds = np.linspace(0.1, 0.9, 17)
    scores = []
    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        scores.append(f1_score(y_true, pred, average="binary", zero_division=0))
    return float(thresholds[int(np.argmax(scores))])

def build_class_weight_dict():
    present_classes = np.unique(y_train)
    weights = np.ones(NUM_CLASSES, dtype=np.float32)
    computed_weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=y_train)
    weights[present_classes] = computed_weights
    return {int(cls): float(weights[cls]) for cls in range(NUM_CLASSES)}

def train_tf_model(
    model_builder,
    model_name: str,
    training_seed: int,
    weight_init_name: str,
    weight_init_tuple,
    epochs: int = 5,
    batch_size: int = 512,
    lr: float = 1e-3,
    use_class_weights: bool = False,
    threshold_tuning: bool = False,
    early_stopping: bool = False,
    scheduler_enabled: bool = False,
):
    set_global_seed(training_seed)
    tf.keras.backend.clear_session()
    model = compile_model(model_builder(weight_init_tuple), lr=lr)

    callbacks = []
    if early_stopping:
        callbacks.append(keras.callbacks.EarlyStopping(monitor="loss", patience=3, restore_best_weights=True))
    if scheduler_enabled:
        callbacks.append(keras.callbacks.ReduceLROnPlateau(monitor="loss", patience=2, factor=0.5, min_lr=1e-5))

    class_weight = build_class_weight_dict() if use_class_weights else None
    model.fit(
        X_train,
        y_train.astype(np.float32 if IS_BINARY else np.int64),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        shuffle=True,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    threshold = 0.5
    if threshold_tuning and IS_BINARY:
        _, train_prob = predict_tf(model, X_train)
        threshold = tune_binary_threshold(y_train, train_prob)

    y_pred, _ = predict_tf(model, X_test, threshold=threshold)
    metrics = evaluate_predictions(y_test, y_pred, labels=list(label_encoder.classes_))
    return {
        "dataset_name": DATASET_NAME,
        "framework": FRAMEWORK,
        "model_name": model_name,
        "training_seed": training_seed,
        "weight_init_name": weight_init_name,
        "weight_init_tuple": json.dumps(weight_init_tuple),
        **RUN_METADATA,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "confusion_matrix": metrics["confusion_matrix"],
        "classification_report": metrics["classification_report"],
    }


import gc

def append_result_row(row: dict, csv_path: Path) -> None:
    row_to_save = row.copy()
    for col in ["confusion_matrix", "classification_report"]:
        if col in row_to_save:
            row_to_save[col] = json.dumps(row_to_save[col])
    pd.DataFrame([row_to_save]).to_csv(
        csv_path,
        mode="a",
        index=False,
        header=not csv_path.exists(),
    )

def cleanup_after_run() -> None:
    gc.collect()
    tf.keras.backend.clear_session()

def run_model_experiment(model_builder, model_name: str, **train_kwargs) -> pd.DataFrame:
    rows = []
    total_runs = len(TRAINING_SEEDS) * len(WEIGHT_INIT_SEEDS)
    run_index = 0
    for training_seed in TRAINING_SEEDS:
        for weight_init_name, weight_init_tuple in WEIGHT_INIT_SEEDS.items():
            run_index += 1
            print(f"[{model_name}] run {run_index}/{total_runs}: seed={training_seed}, init={weight_init_name}")
            row = train_tf_model(
                model_builder=model_builder,
                model_name=model_name,
                training_seed=training_seed,
                weight_init_name=weight_init_name,
                weight_init_tuple=weight_init_tuple,
                **train_kwargs,
            )
            rows.append(row)
            append_result_row(row, RESULTS_CSV_PATH)
            cleanup_after_run()
    return pd.DataFrame(rows)

### Model definitions used by sections 11-14

The framework-style model below is an INSOMNIA-inspired implementation focused on shift-aware IDS evaluation with uncertainty, pseudo-labelling, and adaptation-style regularization; it is not an exact official reproduction unless the original code is provided. The improved MLP is tuned specifically for CICIDS-2017 with class imbalance handling, dropout, batch normalization, early stopping, threshold tuning where applicable, and learning-rate scheduling.

In [ ]:
def build_mlp_baseline(seed_tuple):
    output_units = 1 if IS_BINARY else NUM_CLASSES
    return keras.Sequential([
        layers.Input(shape=(NUM_FEATURES,)),
        layers.Dense(128, activation="relu", kernel_initializer=initializer_from_seed(weight_seed(seed_tuple, 0))),
        layers.Dense(64, activation="relu", kernel_initializer=initializer_from_seed(weight_seed(seed_tuple, 1))),
        layers.Dense(output_units, kernel_initializer=initializer_from_seed(weight_seed(seed_tuple, 2))),
    ])


## Train and evaluate MLP baseline\n\nThis split notebook executes only this model with the configured seed and initialization grid.\n

In [ ]:
if RESULTS_CSV_PATH.exists():
    RESULTS_CSV_PATH.unlink()

per_run_results = run_model_experiment(build_mlp_baseline, "MLP baseline", epochs=5, batch_size=512, lr=1e-3)
per_run_results.head()


## Aggregate and save results\n\nThe per-run CSV is appended after every seed and initialization run. The aggregated CSV is saved at the end.\n

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1"]
required_detail_cols = ["dataset_name", "framework", "model_name", "seed", "weight_init", *metric_cols]
context_cols = [
    col for col in ["held_out_attack"]
    if col in per_run_results.columns and per_run_results[col].notna().any()
]

detailed_results = per_run_results.copy()
detailed_results["seed"] = detailed_results["training_seed"] if "training_seed" in detailed_results.columns else "default"
detailed_results["weight_init"] = detailed_results["weight_init_name"] if "weight_init_name" in detailed_results.columns else "default"
detailed_results = detailed_results[required_detail_cols + context_cols]

aggregation_group_cols = ["dataset_name", "framework", "model_name"] + context_cols
aggregated_results = detailed_results.groupby(aggregation_group_cols, dropna=False)[metric_cols].agg(
    ["mean", "min", "max", "std"]
).round(6)
aggregated_results.columns = [f"{metric}_{stat}" for metric, stat in aggregated_results.columns]
aggregated_results = aggregated_results.reset_index()

DETAILED_RESULTS_PATH = RESULTS_DIR / "detailed_results.csv"
AGGREGATED_RESULTS_PATH = RESULTS_DIR / "aggregated_results.csv"
detailed_results.to_csv(DETAILED_RESULTS_PATH, index=False)
aggregated_results.to_csv(AGGREGATED_RESULTS_PATH, index=False)

print("=== DETAILED RESULTS (PER SEED) ===")
display(detailed_results)

print("=== AGGREGATED RESULTS ===")
display(aggregated_results)

print("Saved detailed results to:", DETAILED_RESULTS_PATH)
print("Saved aggregated results to:", AGGREGATED_RESULTS_PATH)


## Final comparison table\n

In [ ]:
print("=== DONE ===")
